# 03. RAGをゼロから実装：埋め込み→検索→生成

## はじめに：RAGとは何か

**RAG（Retrieval-Augmented Generation）** は、LLMの知識を外部データで補強する技術です。

### RAGパイプラインの流れ

```
【インデックス作成フェーズ】
テキスト → チャンク分割 → 埋め込み（Embedding） → ベクターストア

【クエリフェーズ】
質問 → 埋め込み → 類似検索 → 関連チャンク取得 → LLMで回答生成
```

### RAGが必要な理由
- LLMは学習データにない最新情報を知らない
- 社内ドキュメント・独自データを参照できない
- 根拠となるソースを提示できる（ハルシネーション軽減）

このノートブックで実装するもの：
1. テキストの埋め込みベクトル生成
2. コサイン類似度による意味検索
3. VectorStoreクラスを使った効率的な検索
4. 完全なRAGパイプライン
5. ファイルベースのQ&Aシステム

In [ ]:
import sys
import os
import math
import tempfile

# プロジェクトルートをパスに追加
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.ollama_client import OllamaClient, get_client
from src.rag_utils import VectorStore, RAGPipeline, cosine_similarity

# デフォルトモデル
MODEL = "qwen2.5:7b"
EMBED_MODEL = "nomic-embed-text"  # 埋め込み専用モデル（高速・軽量）

client = get_client(model=MODEL)

if not client.is_available():
    raise RuntimeError("Ollamaサーバーが起動していません。'ollama serve' を実行してください。")

print(f"LLMモデル: {MODEL}")
print(f"埋め込みモデル: {EMBED_MODEL}")
print("セットアップ完了")
print()
print("注: nomic-embed-textが未インストールの場合は以下を実行:")
print("  ollama pull nomic-embed-text")

## 1. 埋め込み（Embedding）とは

**埋め込み（Embedding）** は、テキストを高次元の数値ベクトルに変換する技術です。
意味的に類似したテキストは、ベクトル空間上で近い位置に配置されます。

```
「東京は日本の首都」  → [0.12, -0.34, 0.89, ...] (768次元)
「Japan's capital is Tokyo」→ [0.11, -0.31, 0.87, ...] (意味が近い→ベクトルも近い)
「今日は晴れです」    → [-0.45, 0.67, -0.23, ...] (意味が遠い→ベクトルも遠い)
```

In [ ]:
# 文章の埋め込みを取得
sentence = "東京は日本の首都です。"
print(f"テキスト: '{sentence}'")
print("埋め込みベクトルを生成中...")

embedding = client.embed(sentence, model=EMBED_MODEL)

print(f"\nベクトルの次元数: {len(embedding)}")
print(f"最初の10次元: {[round(v, 4) for v in embedding[:10]]}")
print(f"最後の10次元: {[round(v, 4) for v in embedding[-10:]]}")

# ベクトルの統計
norm = math.sqrt(sum(v**2 for v in embedding))
print(f"\nベクトルのL2ノルム: {norm:.4f}")
print(f"最大値: {max(embedding):.4f}")
print(f"最小値: {min(embedding):.4f}")

# 最初の10次元を簡易可視化
print("\n最初の10次元の値（バーチャート）:")
for i, v in enumerate(embedding[:10]):
    bar_len = int(abs(v) * 20)
    direction = '+' if v >= 0 else '-'
    bar = direction * bar_len
    print(f"  dim[{i:2d}]: {v:+.4f} |{bar}")

In [ ]:
# 複数文章のコサイン類似度行列
sentences = [
    "東京は日本の首都です。",
    "Japan's capital city is Tokyo.",
    "大阪は日本第二の都市です。",
    "機械学習はAIの一分野です。",
    "深層学習はニューラルネットワークを使います。",
    "今日の天気は晴れです。",
]

print("埋め込みベクトルを生成中...")
embeddings = [client.embed(s, model=EMBED_MODEL) for s in sentences]
print(f"{len(sentences)}文の埋め込み完了")

# コサイン類似度行列を計算
print("\n=== コサイン類似度行列 ===")
print("（1.0 = 完全一致, 0.0 = 無関係, -1.0 = 正反対）\n")

# ヘッダー行
short_labels = [f"S{i+1}" for i in range(len(sentences))]
header = "      " + "  ".join(f"{l:>5}" for l in short_labels)
print(header)
print("  " + "-" * (len(header) - 2))

for i, (emb_i, s_i) in enumerate(zip(embeddings, sentences)):
    row = f"{short_labels[i]:>5} |"
    for j, emb_j in enumerate(embeddings):
        sim = cosine_similarity(emb_i, emb_j)
        row += f"  {sim:.3f}"
    print(row)

print()
for i, s in enumerate(sentences):
    print(f"  S{i+1}: {s}")

print("\n観察:")
print("  - S1とS2（東京/Tokyo）は高い類似度 → 同じ意味は言語が違っても近い")
print("  - S4とS5（機械学習/深層学習）は中程度の類似度 → 関連テーマは近い")
print("  - S6（天気）は他と低い類似度 → 無関係なトピックは遠い")

## 2. コサイン類似度による検索

クエリを埋め込みベクトルに変換し、知識ベースの各文書との類似度を計算することで、
意味的に関連する文書を検索できます。

In [ ]:
# 小規模な知識ベースを作成
knowledge_base = [
    "東京は日本の首都で、世界最大の都市圏の一つです。人口は約1400万人。",
    "富士山は日本最高峰の山で、標高3776メートル。静岡県と山梨県の境に位置する。",
    "Pythonは1991年にグイド・ヴァン・ロッサムが開発したプログラミング言語。",
    "機械学習とは、データからパターンを学習してタスクを実行するAIの手法。",
    "大阪は日本第二の都市。たこ焼き・お好み焼きなど食文化で有名。",
    "日本の新幹線は1964年の東海道新幹線開業以来、高速鉄道の象徴となっている。",
    "深層学習（ディープラーニング）は多層ニューラルネットワークを使った機械学習手法。",
    "京都には17件のユネスコ世界文化遺産が存在し、年間5000万人以上が訪れる。",
    "Transformerアーキテクチャは2017年に発表された自然言語処理の革新的なモデル。",
    "北海道は日本最大の島で、広大な農地と豊かな自然が特徴。人口は約500万人。",
]

print(f"知識ベース: {len(knowledge_base)}件の文書")
print("\n埋め込みを生成中...")

kb_embeddings = [client.embed(doc, model=EMBED_MODEL) for doc in knowledge_base]
print("完了")

# 検索関数
def search(query: str, top_k: int = 3) -> list:
    query_emb = client.embed(query, model=EMBED_MODEL)
    scores = [
        (i, cosine_similarity(query_emb, doc_emb))
        for i, doc_emb in enumerate(kb_embeddings)
    ]
    scores.sort(key=lambda x: x[1], reverse=True)
    return [(knowledge_base[i], score) for i, score in scores[:top_k]]

# 検索テスト
queries = [
    "AIと機械学習について教えてください",
    "日本の観光地はどこがいいですか",
    "プログラミング言語について",
]

for query in queries:
    print(f"\nクエリ: '{query}'")
    results = search(query, top_k=3)
    for rank, (doc, score) in enumerate(results, 1):
        print(f"  [{rank}位 類似度:{score:.3f}] {doc[:60]}...")

## 3. VectorStoreを使った検索

`VectorStore` クラスは、埋め込みの管理・検索を効率的に行うためのデータ構造です。
ドキュメントの追加・検索・永続化をシンプルなAPIで操作できます。

In [ ]:
# VectorStoreを使った検索
print("=== VectorStoreの使い方 ===")

# ストアを作成してドキュメントを追加
store = VectorStore(embed_model=EMBED_MODEL, client=client)

# ドキュメントをメタデータ付きで追加
documents = [
    {"text": "東京スカイツリーは高さ634メートルで、日本一高い電波塔。", "category": "観光"},
    {"text": "GPT-4はOpenAIが開発した大規模言語モデル。", "category": "AI"},
    {"text": "日本の人口は約1億2500万人。少子高齢化が課題。", "category": "統計"},
    {"text": "React.jsはFacebook（Meta）が開発したJavaScriptライブラリ。", "category": "技術"},
    {"text": "寿司は江戸時代に発展した日本の伝統料理。", "category": "食文化"},
    {"text": "LLMの推論には大量のGPUメモリが必要。", "category": "AI"},
    {"text": "浅草は東京最古の寺院・浅草寺がある観光エリア。", "category": "観光"},
    {"text": "DockerはコンテナベースのOS仮想化ツール。", "category": "技術"},
]

print(f"\n{len(documents)}件のドキュメントを追加中...")
for doc in documents:
    store.add(doc["text"], metadata={"category": doc["category"]})
print(f"追加完了。ストアのサイズ: {len(store)}件")

# 検索
print("\n--- 検索テスト ---")
query = "AIと大規模言語モデルについて"
print(f"クエリ: '{query}'")
results = store.search(query, top_k=3)

for i, result in enumerate(results, 1):
    text = result.get('text', result.get('content', ''))
    score = result.get('score', result.get('similarity', 0))
    category = result.get('metadata', {}).get('category', '不明')
    print(f"  [{i}位] [{category}] スコア:{score:.3f}")
    print(f"      {text}")

print("\n--- 観光に関する検索 ---")
query2 = "東京の観光スポット"
print(f"クエリ: '{query2}'")
results2 = store.search(query2, top_k=3)
for i, result in enumerate(results2, 1):
    text = result.get('text', result.get('content', ''))
    score = result.get('score', result.get('similarity', 0))
    print(f"  [{i}位] スコア:{score:.3f} - {text}")

## 4. RAGパイプラインの構築

`RAGPipeline` クラスは、埋め込み→検索→生成の全工程を統合したパイプラインです。
テキストを追加してクエリするだけで、コンテキスト付きの回答が得られます。

In [ ]:
# RAGPipelineを使った回答生成
print("=== RAGパイプラインの構築 ===")

pipeline = RAGPipeline(
    client=client,
    model=MODEL,
    embed_model=EMBED_MODEL,
    top_k=3
)

# 知識ベースのテキストを追加
texts = [
    "量子コンピュータは量子力学の原理を利用した次世代コンピュータ。従来のコンピュータより特定問題を高速に解ける。",
    "ChatGPTはOpenAIが2022年11月に公開したチャットボット。GPT-3.5をベースにしている。",
    "日本のGDPは世界第4位（2023年時点）。主要産業は自動車・電子機器・化学。",
    "Rustはメモリ安全性とパフォーマンスを両立したシステムプログラミング言語。2010年にMozillaが開発。",
    "東京オリンピック2020（実際は2021年開催）は57年ぶりの東京開催。COVID-19の影響で無観客となった。",
    "ベクターデータベースは高次元ベクトルの効率的な検索に特化したデータベース。RAGシステムで広く使われる。",
    "日本の平均寿命は男性81歳、女性87歳（2022年）で世界トップクラス。",
    "GitHubはMicrosoftが2018年に買収したコードホスティングプラットフォーム。世界で1億人以上が利用。",
]

print(f"\n{len(texts)}件のテキストをパイプラインに追加中...")
for text in texts:
    pipeline.add_text(text)
print("追加完了")

# クエリを実行
query = "量子コンピュータとは何ですか？"
print(f"\nクエリ: '{query}'")
print("-" * 50)

result = pipeline.query(query)
print(f"RAG回答:\n{result['answer']}")
print("\n参照したコンテキスト:")
for i, ctx in enumerate(result.get('contexts', []), 1):
    ctx_text = ctx if isinstance(ctx, str) else ctx.get('text', ctx.get('content', str(ctx)))
    print(f"  [{i}] {ctx_text[:80]}...")

In [ ]:
# RAGあり vs RAGなし の比較
print("=== RAGあり vs RAGなしの比較 ===")

# RAGで事実を含むテキストを追加
specific_fact = "llmpgプロジェクトはバージョン2.5.0で、2024年3月にリリースされた。主な機能はOllamaクライアント・RAGパイプライン・ベンチマークツールの3つ。"
pipeline.add_text(specific_fact)

query = "llmpgプロジェクトのバージョンと主な機能を教えてください"
print(f"\nクエリ: '{query}'")
print()

# RAGなし（LLMのみ）
print("[RAGなし - LLMの知識のみ]")
response_no_rag = client.generate(
    f"質問: {query}\n\n回答:",
    temperature=0.1
)
print(response_no_rag)

print()
print("-" * 50)
print()

# RAGあり
print("[RAGあり - 知識ベースを参照]")
result_rag = pipeline.query(query)
print(result_rag['answer'])
if result_rag.get('contexts'):
    print("\n[参照ソース]")
    ctx = result_rag['contexts'][0]
    ctx_text = ctx if isinstance(ctx, str) else ctx.get('text', ctx.get('content', str(ctx)))
    print(f"  {ctx_text}")

print()
print("観察: RAGなしはllmpgについて知らないが、RAGありは正確な情報を提供できる")

## 5. より実践的な例：テキストファイルのQ&A

実際のユースケースでは、ファイルから直接テキストを読み込むことが多いです。
`pipeline.add_file()` を使って、テキストファイルをRAGパイプラインに組み込みましょう。

In [ ]:
# サンプルテキストファイルを作成
sample_document = """# AI技術動向レポート 2024年版

## 概要
2024年のAI業界は大規模言語モデル（LLM）の実用化が急速に進んだ年として記録される。
特にローカルLLMの普及により、クラウド依存なしにAIを活用できる環境が整いつつある。

## 主要トレンド

### 1. ローカルLLMの台頭
OllamaやLM Studioなどのツールにより、一般ユーザーでも高性能なLLMをローカル実行できるようになった。
Llama 3、Qwen 2.5、Mistralなどのオープンソースモデルが充実し、GPT-4クラスの性能に近づいている。

### 2. RAGの標準化
企業のAI活用において、RAG（Retrieval-Augmented Generation）が標準的なアーキテクチャとなった。
ChromaDB、Qdrant、Weaviateなどのベクターデータベースが成熟し、導入コストが大幅に低下した。

### 3. エージェント技術の発展
LangChain、AutoGen、CrewAIなどのフレームワークにより、
複数のAIエージェントが協調して複雑なタスクを実行できるマルチエージェントシステムが実用化された。

## 課題と展望

### 課題
- ハルシネーション（事実と異なる情報の生成）の問題は依然として残る
- 長いコンテキストの処理精度向上が必要
- 消費電力・コストの最適化

### 2025年の展望
マルチモーダルAI（テキスト・画像・音声の統合処理）がさらに発展し、
より自然なヒューマンAIインタラクションが実現されると予測される。
特にリアルタイム音声対話やビジョン言語モデルの実用化が加速するだろう。

## 統計データ
- 世界のAI市場規模（2024年）: 約1,842億ドル
- LLMを活用する企業割合（Fortune 500）: 約67%
- 最も人気のオープンソースLLM: Meta Llama 3（GitHub Star数1位）
"""

# 一時ファイルに書き込む
tmp_dir = tempfile.mkdtemp()
doc_path = os.path.join(tmp_dir, "ai_trends_2024.txt")

with open(doc_path, 'w', encoding='utf-8') as f:
    f.write(sample_document)

print(f"サンプルドキュメントを作成: {doc_path}")
print(f"ファイルサイズ: {os.path.getsize(doc_path)} バイト")
print(f"文字数: {len(sample_document)}字")

# 新しいパイプラインでファイルを読み込む
doc_pipeline = RAGPipeline(
    client=client,
    model=MODEL,
    embed_model=EMBED_MODEL,
    top_k=3
)

print("\nファイルをRAGパイプラインに追加中...")
doc_pipeline.add_file(doc_path)
print("完了")

# Q&Aを実行
questions = [
    "2024年のAI業界の主要トレンドを3つ教えてください",
    "RAGとは何ですか？どのように活用されていますか？",
    "LLMの課題は何ですか？",
    "世界のAI市場規模はどれくらいですか？",
]

print("\n=== ドキュメントQ&A ===")
for q in questions:
    print(f"\nQ: {q}")
    result = doc_pipeline.query(q)
    answer = result['answer']
    # 長い回答は先頭200字のみ表示
    print(f"A: {answer[:300]}" + ("..." if len(answer) > 300 else ""))

## まとめ：RAGの仕組みと限界

### RAGパイプラインの全体像

```
インデックス構築:
  テキスト → チャンク分割 → embed() → VectorStore

クエリ処理:
  質問 → embed() → コサイン類似度検索 → Top-K文書取得
       → プロンプト構築（質問 + コンテキスト）→ LLM生成 → 回答
```

### 実装した機能

| コンポーネント | クラス/関数 | 役割 |
|--------------|------------|------|
| 埋め込み生成 | `client.embed()` | テキスト→ベクトル変換 |
| 類似度計算 | `cosine_similarity()` | ベクトル間の意味的距離 |
| ベクターストア | `VectorStore` | 埋め込みの管理・検索 |
| RAGパイプライン | `RAGPipeline` | 全工程の統合 |
| ファイル入力 | `pipeline.add_file()` | テキストファイルの取り込み |

### RAGの限界と対策

| 課題 | 説明 | 対策 |
|------|------|------|
| チャンク品質 | 分割方法で検索精度が変わる | 重複チャンク・階層分割 |
| 埋め込みモデル | 日本語対応が不十分な場合も | 多言語対応モデルの選択 |
| ハルシネーション | コンテキスト外の推測 | プロンプトで制約を追加 |
| スケーラビリティ | 大量文書で遅くなる | 専用ベクターDBの使用 |
| 複雑な推論 | 複数文書の統合が苦手 | Multi-hop RAG・ReRanker |

### 次のステップ
- **ハイブリッド検索**: キーワード検索 + ベクター検索の組み合わせ
- **ReRanker**: 検索結果を再ランキングして精度向上
- **評価指標**: RAGAS・TruLensによるRAG品質評価
- **本格的なベクターDB**: ChromaDB・Qdrant・Weaviateの活用